# Week 1: Gaia Stellar Data Exploration

**Goal:** Query real star data from the Gaia DR3 catalog, explore it raw, clean it, engineer features, and plot the Hertzsprung-Russell (HR) diagram.

**What is Gaia?**  
Gaia is a European Space Agency satellite that has measured the positions, distances, and brightness of ~1.8 billion stars with unprecedented precision. The data is publicly available and queried using ADQL — a dialect of SQL for astronomical archives.

**What is the HR Diagram?**  
A scatter plot of a star's color (temperature proxy) vs. its absolute magnitude (true brightness). Stars don't scatter randomly — they cluster into sequences that reveal their evolutionary stage:  
- **Main sequence** — stars fusing hydrogen in their cores (like our Sun)  
- **Red giant branch** — stars that have exhausted hydrogen and expanded  
- **White dwarfs** — stellar remnants, very hot but intrinsically dim  

---

**Notebook flow:**  
`Setup → Install → Import → Query Gaia → Raw Exploration → Why Cleaning Matters → Clean → Feature Engineering → Visualize`

## Step 0: Install Dependencies

Run this once if `astroquery` is not already installed. Safe to skip if your environment already has it.

In [ ]:
%pip install astroquery

## Step 1: Setup & Imports

We import from `src/` — reusable modules that keep this notebook clean and readable.  
The `sys.path` line tells Python where to find the `src/` folder (one level up from `notebooks/`).

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add the project root to the path so 'from src import ...' works
sys.path.insert(0, os.path.abspath(".."))

from src.data_fetch import fetch_gaia_sample
from src.data_clean import clean_sample, add_features
from src import visualize

## Step 2: Query the Gaia Archive

We fetch 10,000 stars from the `gaiadr3.gaia_source` table using ADQL.

**Why only 10,000?**  
The full DR3 table has ~1.8 billion rows. 10k is enough to see all major stellar populations on the HR diagram while keeping the query fast.

**Key columns we request:**
| Column | What it means |
|--------|---------------|
| `parallax` | Angular shift of the star in milliarcseconds — used to compute distance |
| `parallax_error` | Measurement uncertainty on parallax |
| `phot_g_mean_mag` | Apparent brightness in the G (green) broadband filter |
| `bp_rp` | Color index: Blue minus Red. Low = hot blue star, high = cool red star |

In [ ]:
df = fetch_gaia_sample(top_n=10000)
print(f"Rows returned: {len(df):,}")
df.head()

## Step 3: Raw Exploration — What Does the Data Look Like?

Before any cleaning, explore the raw data to understand its shape and spot problems.

In [ ]:
# Summary statistics on the raw columns we care about
print(df[["parallax", "parallax_error", "phot_g_mean_mag", "bp_rp"]].describe())

In [ ]:
# Raw parallax distribution — notice the negative values on the left
# Negative parallax is unphysical and results from measurement noise on very distant stars
df["parallax"].hist(bins=100)
plt.xlabel("Parallax (mas)")
plt.ylabel("Number of Stars")
plt.title("Raw Parallax Distribution (before cleaning)")
plt.show()

### Why Cleaning Matters — The Negative Distance Problem

Distance is computed as `d = 1000 / parallax`. Let's naively apply it to the raw data to see what goes wrong:

In [ ]:
# Compute distance on RAW uncleaned data — deliberately showing the problem
df["distance_pc"] = 1000 / df["parallax"]

# Print first 10 rows — notice some negative distances (row 4 is a good example)
print(df[["parallax", "parallax_error", "distance_pc"]].head(10))
print(f"\nStars with negative distance:        {(df['distance_pc'] < 0).sum()}")
print(f"Stars with distance > 10,000 pc (noise): {(df['distance_pc'] > 10000).sum()}")

**This is why we clean.** Negative distances and wildly uncertain measurements corrupt the HR diagram. Each filter in the next step has a specific physical motivation.

## Step 4: Clean the Data

| Filter | What it removes | Why |
|--------|----------------|-----|
| `parallax > 0` | Negative parallax | Unphysical — distance would be negative |
| `parallax_error > 0` | Zero-error rows | Prevents division by zero in SNR calculation |
| `parallax / parallax_error > 5` | Low signal-to-noise stars | Distance uncertainty > 25% → absolute magnitude unreliable |
| `bp_rp` not null | Stars with no color | Can't be placed on the HR diagram |

Full implementation and comments: [src/data_clean.py](../src/data_clean.py)

In [ ]:
clean = clean_sample(df)

In [ ]:
# Parallax distribution after cleaning — negative values gone, noisy outliers removed
clean["parallax"].hist(bins=100, color="steelblue", edgecolor="none")
plt.xlabel("Parallax (mas)")
plt.ylabel("Number of Stars")
plt.title("Parallax Distribution (after cleaning)")
plt.show()

## Step 5: Feature Engineering

Two derived columns needed for the HR diagram — neither comes directly from Gaia:

**Distance in parsecs** (`distance_pc`):  
`d = 1000 / parallax_mas` — 1 parsec ≈ 3.26 light-years. Parallax in milliarcseconds inverts directly.

**Absolute magnitude** (`absolute_mag`):  
`M = m − 5 × log₁₀(d / 10)` — the **distance modulus** formula. Corrects apparent brightness for distance to give a star's true intrinsic luminosity. At d = 10 pc, M = m (the standard reference point).

Full implementation: [src/data_clean.py](../src/data_clean.py)

In [ ]:
clean = add_features(clean)
print(clean[["bp_rp", "absolute_mag", "distance_pc"]].describe())

In [ ]:
# Sanity check: our Sun has absolute magnitude ~4.83 and bp_rp ~0.82
# Stars matching these values are solar analogues — a good sanity check on the pipeline
solar_analogues = clean[
    (clean["absolute_mag"].between(4.0, 5.5)) &
    (clean["bp_rp"].between(0.6, 1.0))
]
print(f"Solar analogue candidates in sample: {len(solar_analogues)}")
print(solar_analogues[["bp_rp", "absolute_mag", "distance_pc"]].head())

## Step 6: Visualize

### 6a. HR Diagram — Density Map

Hex-bin plot with log-color scale reveals structure across orders of magnitude in star count.  
Y-axis is **inverted** — astronomy convention: bright (low magnitude number) at the top.

**What to look for:**
- **Bright diagonal band** top-left → bottom-right: **main sequence**
- **Clump upper-right** (bright + red): **red giants**
- **Sparse dots lower-left** (dim + blue): **white dwarfs**

In [ ]:
visualize.plot_hr_density(clean)

### 6b. Absolute Magnitude Distribution

The peak is dominated by main-sequence stars around M ≈ 4–6. Our Sun sits at M ≈ 4.83 — right in the middle of the most common stellar brightness range.

In [ ]:
visualize.plot_abs_mag_hist(clean)

### 6c. Distance vs. Brightness — Malmquist Bias

**Malmquist bias:** at larger distances, only intrinsically bright stars are above Gaia's detection limit. Faint distant stars are invisible — so this sample is biased toward luminous stars, not a true census.

The missing lower-right corner in the plot (faint + far = undetectable) makes the bias visible.

In [ ]:
visualize.plot_distance_vs_mag(clean)

---

## Summary

| Step | What we did | Key insight |
|------|-------------|-------------|
| Query | Fetched 10k stars from Gaia DR3 via ADQL | Real observational data, not simulated |
| Raw explore | Parallax distribution + raw distances | Negative distances show *why* cleaning is necessary |
| Clean | Removed negative parallax, zero-error, low-SNR, missing color | Each filter has a specific physical motivation |
| Features | Computed `distance_pc` and `absolute_mag` | Distance modulus corrects for how far away stars are |
| Sanity check | Found solar analogues in the sample | Validates the feature engineering is correct |
| Visualize | HR diagram, magnitude histogram, Malmquist plot | Sample biases are real and visible in the data |

**Next:** [`week2_kmeans_clustering.ipynb`](week2_kmeans_clustering.ipynb) — use K-Means to recover stellar populations from the HR diagram without any labels.